# RAG Chunk Size Experiment — Retrieval-Only Evaluation

**Objective:** Measure how `chunk_size` and `overlap` affect retrieval quality using standard IR metrics — no LLM generation required, $0 API cost.

| Variable | Values |
|---|---|
| Chunk sizes (tokens) | 128, 256, 512, 1024, 2048 |
| Overlap | 0%, 10%, 20% of chunk size |
| **Total combinations** | **15** |

**Metrics:**

| Metric | Definition |
|---|---|
| **Hit Rate @ K** | Fraction of queries where ≥1 retrieved chunk contains the ground-truth answer |
| **MRR @ K** | Mean Reciprocal Rank — `1/rank` of the first relevant chunk (0 if none in top-K) |
| **Avg retrieval latency** | Wall-clock time for embedding query + vector search |
| **Avg chunks per doc** | Index size indicator |

**Relevance criterion:** a chunk is considered relevant if it contains the ground-truth answer string (case-insensitive substring match). This is exact for SQuAD since answers are verbatim spans from the context.

**Stack:** LangChain · ChromaDB · `nomic-embed-text` via Ollama (all local, no API keys needed)

## 0. Install dependencies

Run once, then comment out.

In [ ]:
# !pip install \
#     langchain==0.3.25 \
#     langchain-community==0.3.24 \
#     langchain-chroma==0.2.4 \
#     langchain-ollama==0.3.3 \
#     chromadb==0.6.3 \
#     datasets==3.6.0 \
#     pandas==2.2.3 \
#     seaborn==0.13.2 \
#     matplotlib==3.10.3 \
#     tqdm==4.67.1

## 1. Imports & configuration

In [ ]:
import time
import warnings
import itertools
from pathlib import Path
from typing import List, Dict, Any

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document
from langchain_community.vectorstores import Chroma
from langchain_ollama import OllamaEmbeddings
from datasets import load_dataset

warnings.filterwarnings("ignore")

# ── Paths ─────────────────────────────────────────────────────────────────────
BASE_DIR    = Path(".").resolve()
FIGURES_DIR = BASE_DIR / "figures"
FIGURES_DIR.mkdir(exist_ok=True)
RESULTS_CSV = BASE_DIR / "results.csv"

# ── Experiment grid ────────────────────────────────────────────────────────────
CHUNK_SIZES    = [128, 256, 512, 1024, 2048]   # tokens (≈ chars × 0.25)
OVERLAP_RATIOS = [0.0, 0.10, 0.20]             # fraction of chunk_size
N_SAMPLES      = 200                           # Q&A pairs from SQuAD
TOP_K          = 3                             # chunks retrieved per query

# ── Embedding model ───────────────────────────────────────────────────────────
EMBED_MODEL = "nomic-embed-text"               # Ollama local model
OLLAMA_BASE = "http://localhost:11434"

# nomic-embed-text uses roughly 4 chars per token
CHARS_PER_TOKEN = 4

print("✅ Configuration loaded — no API keys required.")
print(f"   Grid: {len(CHUNK_SIZES)} chunk sizes × {len(OVERLAP_RATIOS)} overlaps = {len(CHUNK_SIZES)*len(OVERLAP_RATIOS)} combos")
print(f"   Samples: {N_SAMPLES}   Top-K: {TOP_K}")

## 2. Load SQuAD dataset

In [ ]:
def load_squad_samples(n: int = 200, seed: int = 42) -> List[Dict]:
    """Load n Q&A pairs from SQuAD validation split.
    Each sample: {question, ground_truth, context}
    ground_truth is a verbatim span from context — perfect for substring matching.
    """
    ds = load_dataset("rajpurkar/squad", split="validation")
    ds = ds.shuffle(seed=seed).select(range(n))

    return [
        {
            "question":     row["question"],
            "ground_truth": row["answers"]["text"][0],  # first accepted answer
            "context":      row["context"],
        }
        for row in ds
    ]

squad_samples = load_squad_samples(N_SAMPLES)

print(f"Loaded {len(squad_samples)} samples.")
ex = squad_samples[0]
print(f"\nExample:")
print(f"  Question:     {ex['question']}")
print(f"  Ground truth: {ex['ground_truth']}")
print(f"  Context len:  {len(ex['context'])} chars")
print(f"\n  ✓ Answer in context: {ex['ground_truth'].lower() in ex['context'].lower()}")

## 3. Helper functions

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 3a. Build in-memory ChromaDB for one (chunk_size, overlap_ratio) combo
# ─────────────────────────────────────────────────────────────────────────────
def build_vectorstore(
    samples: List[Dict],
    chunk_size: int,
    overlap_ratio: float,
) -> tuple[Chroma, int]:
    """Returns (vectorstore, total_chunk_count)."""
    chunk_chars   = chunk_size * CHARS_PER_TOKEN
    overlap_chars = int(chunk_chars * overlap_ratio)

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_chars,
        chunk_overlap=overlap_chars,
        separators=["\n\n", "\n", ". ", " ", ""],
    )

    docs: List[Document] = []
    for i, s in enumerate(samples):
        for j, chunk in enumerate(splitter.split_text(s["context"])):
            docs.append(Document(
                page_content=chunk,
                metadata={"sample_idx": i, "chunk_idx": j},
            ))

    embeddings = OllamaEmbeddings(model=EMBED_MODEL, base_url=OLLAMA_BASE)

    vs = Chroma.from_documents(
        documents=docs,
        embedding=embeddings,
        collection_name=f"rag_cs{chunk_size}_ov{int(overlap_ratio*100)}",
    )
    return vs, len(docs)


# ─────────────────────────────────────────────────────────────────────────────
# 3b. Retrieval metrics (pure Python, no LLM)
# ─────────────────────────────────────────────────────────────────────────────
def is_relevant(chunk_text: str, ground_truth: str) -> bool:
    """True if ground_truth appears (case-insensitive) in chunk_text.
    Valid for SQuAD because answers are verbatim context spans.
    """
    return ground_truth.strip().lower() in chunk_text.lower()


def hit_rate(retrieved: List[str], ground_truth: str) -> float:
    """1.0 if any retrieved chunk is relevant, else 0.0."""
    return float(any(is_relevant(c, ground_truth) for c in retrieved))


def reciprocal_rank(retrieved: List[str], ground_truth: str) -> float:
    """1/rank of the first relevant chunk (0.0 if none found)."""
    for rank, chunk in enumerate(retrieved, start=1):
        if is_relevant(chunk, ground_truth):
            return 1.0 / rank
    return 0.0


print("✅ Helper functions defined.")

## 4. Main experiment loop

> ⏱ **Expected runtime:** ~10–20 min (embedding is the bottleneck; no generation calls).

In [ ]:
def run_combination(
    samples: List[Dict],
    chunk_size: int,
    overlap_ratio: float,
    top_k: int = TOP_K,
) -> Dict[str, Any]:
    combo_label = f"cs={chunk_size}, ov={int(overlap_ratio*100)}%"
    print(f"\n{'─'*55}")
    print(f"  {combo_label}")
    print(f"{'─'*55}")

    # ── Build index ───────────────────────────────────────────────────────────
    print("  [1/2] Building vector store...")
    t0 = time.perf_counter()
    vs, total_chunks = build_vectorstore(samples, chunk_size, overlap_ratio)
    build_time = time.perf_counter() - t0
    avg_chunks_per_doc = total_chunks / len(samples)
    print(f"        {total_chunks} chunks in {build_time:.1f}s  (avg {avg_chunks_per_doc:.1f}/doc)")

    # ── Retrieve & evaluate ───────────────────────────────────────────────────
    print(f"  [2/2] Retrieving for {len(samples)} queries...")
    retriever  = vs.as_retriever(search_kwargs={"k": top_k})
    hit_rates  = []
    mrr_scores = []
    latencies  = []

    for s in tqdm(samples, desc=combo_label, leave=False):
        t_q = time.perf_counter()
        docs = retriever.invoke(s["question"])
        latencies.append(time.perf_counter() - t_q)

        chunks = [d.page_content for d in docs]
        hit_rates.append(hit_rate(chunks, s["ground_truth"]))
        mrr_scores.append(reciprocal_rank(chunks, s["ground_truth"]))

    result = {
        "chunk_size":          chunk_size,
        "overlap_ratio":       overlap_ratio,
        "overlap_tokens":      int(chunk_size * overlap_ratio),
        "hit_rate":            float(np.mean(hit_rates)),
        "mrr":                 float(np.mean(mrr_scores)),
        "avg_latency_s":       float(np.mean(latencies)),
        "p95_latency_s":       float(np.percentile(latencies, 95)),
        "avg_chunks_per_doc":  avg_chunks_per_doc,
        "total_chunks":        total_chunks,
        "build_time_s":        build_time,
        "n_samples":           len(samples),
        "top_k":               top_k,
    }

    print(f"  ✓ hit_rate@{top_k}={result['hit_rate']:.3f}  "
          f"mrr@{top_k}={result['mrr']:.3f}  "
          f"avg_latency={result['avg_latency_s']:.2f}s")

    vs._client.delete_collection(vs._collection.name)
    return result

In [ ]:
# ── Resume: skip combos already in results.csv ────────────────────────────────
completed_keys = set()
existing_df    = pd.DataFrame()
if RESULTS_CSV.exists():
    existing_df    = pd.read_csv(RESULTS_CSV)
    completed_keys = set(zip(existing_df["chunk_size"], existing_df["overlap_ratio"]))
    print(f"Resuming: {len(completed_keys)} combos already complete.")

all_results = []

for chunk_size, overlap_ratio in itertools.product(CHUNK_SIZES, OVERLAP_RATIOS):
    if (chunk_size, overlap_ratio) in completed_keys:
        print(f"⏩ Skipping cs={chunk_size}, ov={overlap_ratio}")
        continue

    result = run_combination(squad_samples, chunk_size, overlap_ratio)
    all_results.append(result)

    # Checkpoint
    new_df = pd.DataFrame(all_results)
    if not existing_df.empty:
        new_df = pd.concat([existing_df, new_df], ignore_index=True)
    new_df.to_csv(RESULTS_CSV, index=False)
    print(f"  💾 Checkpointed → {RESULTS_CSV}")

print("\n🎉 All 15 combinations complete!")

## 5. Load & inspect results

In [ ]:
df = pd.read_csv(RESULTS_CSV)
df["overlap_pct"] = (df["overlap_ratio"] * 100).astype(int).astype(str) + "%"

display_cols = [
    "chunk_size", "overlap_pct",
    "hit_rate", "mrr",
    "avg_latency_s", "avg_chunks_per_doc",
]

df[display_cols].sort_values(["chunk_size", "overlap_pct"]).style \
    .background_gradient(subset=["hit_rate", "mrr"], cmap="YlGn") \
    .background_gradient(subset=["avg_latency_s"], cmap="YlOrRd_r") \
    .format({
        "hit_rate":          "{:.3f}",
        "mrr":               "{:.3f}",
        "avg_latency_s":     "{:.2f}s",
        "avg_chunks_per_doc":"{:.1f}",
    })

## 6. Visualizations

In [ ]:
plt.rcParams.update({
    "figure.dpi":         150,
    "font.family":        "sans-serif",
    "axes.spines.top":    False,
    "axes.spines.right":  False,
})
OVERLAP_ORDER = ["0%", "10%", "20%"]


# ── 6.1  Heatmap: Hit Rate ────────────────────────────────────────────────────
def plot_heatmap(metric: str, title: str, filename: str,
                 vmin: float = 0.0, vmax: float = 1.0, cmap: str = "YlGn"):
    pivot = df.pivot_table(values=metric, index="chunk_size", columns="overlap_pct")[OVERLAP_ORDER]
    fig, ax = plt.subplots(figsize=(7, 4))
    sns.heatmap(
        pivot, annot=True, fmt=".3f",
        cmap=cmap, linewidths=0.5,
        vmin=vmin, vmax=vmax, ax=ax,
    )
    ax.set_title(title, fontsize=13, fontweight="bold", pad=12)
    ax.set_xlabel("Overlap", fontsize=11)
    ax.set_ylabel("Chunk size (tokens)", fontsize=11)
    plt.tight_layout()
    path = FIGURES_DIR / filename
    fig.savefig(path, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {path}")

plot_heatmap("hit_rate", f"Hit Rate @ {TOP_K}  (chunk × overlap)", "heatmap_hit_rate.png")
plot_heatmap("mrr",      f"MRR @ {TOP_K}  (chunk × overlap)",      "heatmap_mrr.png")

In [ ]:
# ── 6.2  Latency line chart ───────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
palette = sns.color_palette("Set2", len(OVERLAP_ORDER))

for color, ov in zip(palette, OVERLAP_ORDER):
    sub = df[df["overlap_pct"] == ov].sort_values("chunk_size")
    ax.plot(
        sub["chunk_size"], sub["avg_latency_s"],
        marker="o", linewidth=2, color=color, label=f"Overlap {ov}",
    )
    ax.fill_between(
        sub["chunk_size"], sub["avg_latency_s"], sub["p95_latency_s"],
        alpha=0.15, color=color,
    )

ax.set_xscale("log", base=2)
ax.set_xticks(CHUNK_SIZES)
ax.set_xticklabels([str(c) for c in CHUNK_SIZES])
ax.set_xlabel("Chunk size (tokens)", fontsize=11)
ax.set_ylabel("Avg retrieval latency (s)", fontsize=11)
ax.set_title("Retrieval Latency vs Chunk Size  (shaded = avg → p95)",
             fontsize=13, fontweight="bold")
ax.legend()
plt.tight_layout()
path = FIGURES_DIR / "latency_vs_chunksize.png"
fig.savefig(path, bbox_inches="tight")
plt.show()
print(f"  Saved: {path}")

In [ ]:
# ── 6.3  Hit Rate & MRR vs chunk size (line, both metrics together) ───────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=False)
palette = sns.color_palette("Set2", len(OVERLAP_ORDER))

for ax, metric, title in zip(
    axes,
    ["hit_rate", "mrr"],
    [f"Hit Rate @ {TOP_K}", f"MRR @ {TOP_K}"],
):
    for color, ov in zip(palette, OVERLAP_ORDER):
        sub = df[df["overlap_pct"] == ov].sort_values("chunk_size")
        ax.plot(
            sub["chunk_size"], sub[metric],
            marker="o", linewidth=2, color=color, label=f"Overlap {ov}",
        )
    ax.set_xscale("log", base=2)
    ax.set_xticks(CHUNK_SIZES)
    ax.set_xticklabels([str(c) for c in CHUNK_SIZES])
    ax.set_xlabel("Chunk size (tokens)", fontsize=11)
    ax.set_ylabel(title, fontsize=11)
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=9)

plt.suptitle("Retrieval Quality vs Chunk Size", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
path = FIGURES_DIR / "retrieval_quality_vs_chunksize.png"
fig.savefig(path, bbox_inches="tight")
plt.show()
print(f"  Saved: {path}")

## 7. Key findings

In [ ]:
best_hr      = df.loc[df["hit_rate"].idxmax()]
best_mrr     = df.loc[df["mrr"].idxmax()]
fastest      = df.loc[df["avg_latency_s"].idxmin()]
best_overall = df.loc[(df["hit_rate"] + df["mrr"]).idxmax()]  # equal-weight composite

print(f"{'Metric':<28} {'Chunk':>6} {'Overlap':>9} {'Value':>8}")
print("─" * 58)
rows = [
    (f"Best Hit Rate@{TOP_K}",     best_hr,      "hit_rate",      "{:.3f}"),
    (f"Best MRR@{TOP_K}",          best_mrr,     "mrr",           "{:.3f}"),
    ("Fastest (avg latency)",       fastest,      "avg_latency_s", "{:.2f}s"),
    ("Best overall (HR+MRR)",       best_overall, "hit_rate",      "{:.3f}"),
]
for label, row, col, fmt in rows:
    print(f"{label:<28} {int(row['chunk_size']):>6} {row['overlap_pct']:>9} {fmt.format(row[col]):>8}")

print("\n─── Trade-off summary ────────────────────────────────────────")
correlation = df[["chunk_size", "hit_rate", "mrr", "avg_latency_s"]].corr()["chunk_size"]
print("Pearson correlation with chunk_size:")
for col in ["hit_rate", "mrr", "avg_latency_s"]:
    print(f"  {col:<20} r={correlation[col]:+.3f}")

## 8. Save final results

In [ ]:
df.to_csv(RESULTS_CSV, index=False)
print(f"✅ Results saved → {RESULTS_CSV}")
print(f"   Shape: {df.shape}")
print("\nAll output files:")
for f in sorted(list(BASE_DIR.glob("*.csv")) + list(FIGURES_DIR.glob("*.png"))):
    print(f"  {f.relative_to(BASE_DIR)}")

---
*Experiment by Dalai Aguirre — setup documented in `README.md`*

## 9. Embeddings Experiment: `sentence-transformers/all-MiniLM-L6-v2`

Same 5 × 3 grid as the TF-IDF run, but with dense semantic embeddings instead of sparse bag-of-words.

| Setting | Value |
|---|---|
| Embedding model | `sentence-transformers/all-MiniLM-L6-v2` |
| Embedding dim | 384 |
| Vector store | ChromaDB (ephemeral, in-memory) |
| Similarity | Cosine |
| Text splitter | Recursive character splitter (same logic as Section 3) |
| Output file | `results_embeddings.csv` |

> ⏱ **Expected runtime:** ~3–8 min on CPU (model is lightweight, no API calls).

In [ ]:

import time, itertools, warnings
from pathlib import Path
from typing import List, Dict, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
import chromadb

warnings.filterwarnings("ignore")

# ── Paths & constants ──────────────────────────────────────────────────────────
BASE_DIR          = Path(".").resolve()
FIGURES_DIR       = BASE_DIR / "figures"
FIGURES_DIR.mkdir(exist_ok=True)
RESULTS_EMB_CSV   = BASE_DIR / "results_embeddings.csv"
RESULTS_TFIDF_CSV = BASE_DIR / "results.csv"

CHUNK_SIZES      = [128, 256, 512, 1024, 2048]
OVERLAP_RATIOS   = [0.0, 0.10, 0.20]
N_SAMPLES        = 200
TOP_K            = 3
CHARS_PER_TOKEN  = 4
EMB_MODEL_NAME   = "all-MiniLM-L6-v2"
RETRIEVAL_METHOD = f"sentence-transformers/{EMB_MODEL_NAME}"

# ── Load model once (~90 MB on first download, cached after that) ──────────────
print(f"Loading {EMB_MODEL_NAME} ...")
embed_model = SentenceTransformer(EMB_MODEL_NAME)
print(f"Model ready  —  dim={embed_model.get_sentence_embedding_dimension()}")
print(f"   Grid: {len(CHUNK_SIZES)} x {len(OVERLAP_RATIOS)} = "
      f"{len(CHUNK_SIZES)*len(OVERLAP_RATIOS)} combos | Samples: {N_SAMPLES} | Top-K: {TOP_K}")


# ── Text splitter ──────────────────────────────────────────────────────────────
def split_text(text: str, chunk_chars: int, overlap_chars: int) -> List[str]:
    chunks, start = [], 0
    while True:
        end   = min(start + chunk_chars, len(text))
        chunk = text[start:end]
        cut   = end          # nominal end; may be trimmed to a sentence boundary
        if end < len(text):
            for sep in ["\n\n", "\n", ". ", " "]:
                idx = chunk.rfind(sep)
                if idx > chunk_chars // 2:
                    chunk = chunk[:idx + len(sep)]
                    cut   = start + idx + len(sep)
                    break
        if chunk.strip():
            chunks.append(chunk.strip())
        if end == len(text):
            break                          # reached end of text — stop
        start = max(cut - overlap_chars, start + 1)
    return chunks


# ── Build ephemeral ChromaDB for one grid cell ─────────────────────────────────
def build_vectorstore_emb(
    samples: List[Dict], chunk_size: int, overlap_ratio: float
) -> tuple:
    chunk_chars   = chunk_size * CHARS_PER_TOKEN
    overlap_chars = int(chunk_chars * overlap_ratio)

    all_chunks, all_ids = [], []
    for i, s in enumerate(samples):
        for j, chunk in enumerate(split_text(s["context"], chunk_chars, overlap_chars)):
            all_chunks.append(chunk)
            all_ids.append(f"s{i}_c{j}")

    embeddings = embed_model.encode(all_chunks, batch_size=64, show_progress_bar=False)

    client = chromadb.EphemeralClient()
    col    = client.create_collection("exp", metadata={"hnsw:space": "cosine"})
    batch  = 5000
    for i in range(0, len(all_chunks), batch):
        col.add(
            documents=all_chunks[i:i+batch],
            embeddings=embeddings[i:i+batch].tolist(),
            ids=all_ids[i:i+batch],
        )
    return col, client, len(all_chunks)


# ── Retrieval metrics ──────────────────────────────────────────────────────────
def _relevant(chunk: str, gt: str) -> bool:
    return gt.strip().lower() in chunk.lower()

def hit_rate_emb(chunks: List[str], gt: str) -> float:
    return float(any(_relevant(c, gt) for c in chunks))

def mrr_emb(chunks: List[str], gt: str) -> float:
    for rank, c in enumerate(chunks, 1):
        if _relevant(c, gt):
            return 1.0 / rank
    return 0.0


# ── Single-combination runner ──────────────────────────────────────────────────
def run_combination_emb(
    samples: List[Dict], chunk_size: int, overlap_ratio: float, top_k: int = TOP_K
) -> Dict[str, Any]:
    label = f"cs={chunk_size}, ov={int(overlap_ratio*100)}%"
    print(f"\n{'-'*55}\n  {label}\n{'-'*55}")

    print("  [1/2] Building vector store (encoding all chunks)...")
    t0 = time.perf_counter()
    col, client, total_chunks = build_vectorstore_emb(samples, chunk_size, overlap_ratio)
    build_time         = time.perf_counter() - t0
    avg_chunks_per_doc = total_chunks / len(samples)
    print(f"        {total_chunks} chunks in {build_time:.1f}s  (avg {avg_chunks_per_doc:.1f}/doc)")

    print(f"  [2/2] Retrieving for {len(samples)} queries...")
    hit_rates, mrr_scores, latencies = [], [], []
    q_embs = embed_model.encode([s["question"] for s in samples], batch_size=64,
                                show_progress_bar=False)

    for s, q_emb in tqdm(zip(samples, q_embs), total=len(samples), desc=label, leave=False):
        t_q = time.perf_counter()
        res = col.query(query_embeddings=[q_emb.tolist()], n_results=top_k)
        latencies.append(time.perf_counter() - t_q)
        retrieved = res["documents"][0]
        hit_rates.append(hit_rate_emb(retrieved, s["ground_truth"]))
        mrr_scores.append(mrr_emb(retrieved, s["ground_truth"]))

    result = {
        "chunk_size":         chunk_size,
        "overlap_ratio":      overlap_ratio,
        "overlap_tokens":     int(chunk_size * overlap_ratio),
        "hit_rate":           float(np.mean(hit_rates)),
        "mrr":                float(np.mean(mrr_scores)),
        "avg_latency_s":      float(np.mean(latencies)),
        "p95_latency_s":      float(np.percentile(latencies, 95)),
        "avg_chunks_per_doc": avg_chunks_per_doc,
        "total_chunks":       total_chunks,
        "build_time_s":       build_time,
        "n_samples":          len(samples),
        "top_k":              top_k,
        "retrieval_method":   RETRIEVAL_METHOD,
    }
    print(f"  OK hit_rate@{top_k}={result['hit_rate']:.3f}  "
          f"mrr@{top_k}={result['mrr']:.3f}  "
          f"avg_latency={result['avg_latency_s']:.4f}s")

    client.delete_collection("exp")
    return result


print("\nAll helpers defined.")


In [ ]:

# ── Load SQuAD (reuse from Section 2 if still in scope) ───────────────────────
try:
    assert len(squad_samples) == N_SAMPLES
    print(f"Reusing {len(squad_samples)} SQuAD samples already in memory.")
except (NameError, AssertionError):
    _ds = load_dataset("rajpurkar/squad", split="validation")
    _ds = _ds.shuffle(seed=42).select(range(N_SAMPLES))
    squad_samples = [
        {
            "question":     row["question"],
            "ground_truth": row["answers"]["text"][0],
            "context":      row["context"],
        }
        for row in _ds
    ]
    print(f"Loaded {len(squad_samples)} SQuAD samples.")

# ── Resume support ─────────────────────────────────────────────────────────────
_completed_emb = set()
_existing_emb  = pd.DataFrame()
if RESULTS_EMB_CSV.exists():
    _existing_emb  = pd.read_csv(RESULTS_EMB_CSV)
    _completed_emb = set(zip(_existing_emb["chunk_size"], _existing_emb["overlap_ratio"]))
    print(f"Resuming: {len(_completed_emb)} combos already done.")

# ── Main grid loop ─────────────────────────────────────────────────────────────
_results_emb = []

for chunk_size, overlap_ratio in itertools.product(CHUNK_SIZES, OVERLAP_RATIOS):
    if (chunk_size, overlap_ratio) in _completed_emb:
        print(f"⏩ Skipping cs={chunk_size}, ov={overlap_ratio}")
        continue

    result = run_combination_emb(squad_samples, chunk_size, overlap_ratio)
    _results_emb.append(result)

    _new_df = pd.DataFrame(_results_emb)
    if not _existing_emb.empty:
        _new_df = pd.concat([_existing_emb, _new_df], ignore_index=True)
    _new_df.to_csv(RESULTS_EMB_CSV, index=False)
    print(f"  Checkpointed → {RESULTS_EMB_CSV}")

print(f"\n🎉 All {len(CHUNK_SIZES)*len(OVERLAP_RATIOS)} combinations complete!")
print(f"   Results saved → {RESULTS_EMB_CSV}")


## 10. Results & Comparison: Embeddings vs TF-IDF

In [ ]:

# ── Load both result files ─────────────────────────────────────────────────────
df_emb   = pd.read_csv(RESULTS_EMB_CSV)
df_tfidf = pd.read_csv(RESULTS_TFIDF_CSV)

df_emb["method"]   = "sentence-transformers"
df_tfidf["method"] = "TF-IDF"
for _df in [df_emb, df_tfidf]:
    _df["overlap_pct"] = (_df["overlap_ratio"] * 100).astype(int).astype(str) + "%"

df_all        = pd.concat([df_tfidf, df_emb], ignore_index=True)
OVERLAP_ORDER = ["0%", "10%", "20%"]
METHODS       = ["TF-IDF", "sentence-transformers"]

plt.rcParams.update({"figure.dpi": 150, "font.family": "sans-serif",
                     "axes.spines.top": False, "axes.spines.right": False})

# ── 10.1  Embeddings result table ──────────────────────────────────────────────
print("=== Embeddings results ===")
_display_cols = ["chunk_size", "overlap_pct", "hit_rate", "mrr",
                 "avg_latency_s", "avg_chunks_per_doc"]
display(df_emb[_display_cols].sort_values(["chunk_size", "overlap_pct"]).style
    .background_gradient(subset=["hit_rate", "mrr"], cmap="YlGn")
    .background_gradient(subset=["avg_latency_s"], cmap="YlOrRd_r")
    .format({"hit_rate": "{:.3f}", "mrr": "{:.3f}",
             "avg_latency_s": "{:.4f}s", "avg_chunks_per_doc": "{:.1f}"}))

# ── 10.2  Comparison heatmaps (2×2: method × metric) ──────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for col_idx, method in enumerate(METHODS):
    _dfm = df_all[df_all["method"] == method]
    for row_idx, metric in enumerate(["hit_rate", "mrr"]):
        ax    = axes[row_idx][col_idx]
        pivot = _dfm.pivot_table(values=metric, index="chunk_size", columns="overlap_pct")[OVERLAP_ORDER]
        sns.heatmap(pivot, annot=True, fmt=".3f", cmap="YlGn",
                    linewidths=0.5, vmin=0.0, vmax=1.0, ax=ax)
        _lbl = "Hit Rate" if metric == "hit_rate" else "MRR"
        ax.set_title(f"{method}  —  {_lbl} @ {TOP_K}", fontsize=11, fontweight="bold")
        ax.set_xlabel("Overlap", fontsize=10)
        ax.set_ylabel("Chunk size (tokens)", fontsize=10)

plt.suptitle("TF-IDF vs Sentence-Transformers: Retrieval Quality",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
_path = FIGURES_DIR / "comparison_heatmaps.png"
fig.savefig(_path, bbox_inches="tight")
plt.show()
print(f"Saved: {_path}")

# ── 10.3  Line chart: metric vs chunk size (overlap=0%) ───────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
_palette = {"TF-IDF": "#4C72B0", "sentence-transformers": "#DD8452"}
_markers = {"TF-IDF": "o", "sentence-transformers": "s"}

for ax, metric in zip(axes, ["hit_rate", "mrr"]):
    for method in METHODS:
        sub = (df_all[(df_all["method"] == method) & (df_all["overlap_pct"] == "0%")]
               .sort_values("chunk_size"))
        ax.plot(sub["chunk_size"], sub[metric],
                marker=_markers[method], linewidth=2,
                color=_palette[method], label=method)
    ax.set_xscale("log", base=2)
    ax.set_xticks(CHUNK_SIZES)
    ax.set_xticklabels([str(c) for c in CHUNK_SIZES])
    ax.set_xlabel("Chunk size (tokens)", fontsize=11)
    _lbl = "Hit Rate" if metric == "hit_rate" else "MRR"
    ax.set_ylabel(f"{_lbl} @ {TOP_K}", fontsize=11)
    ax.set_title(f"{_lbl} @ {TOP_K}  (overlap = 0%)", fontsize=12, fontweight="bold")
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=10)

plt.suptitle("TF-IDF vs Sentence-Transformers (no overlap)",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
_path = FIGURES_DIR / "comparison_quality.png"
fig.savefig(_path, bbox_inches="tight")
plt.show()
print(f"Saved: {_path}")

# ── 10.4  Delta heatmap: embeddings − TF-IDF ──────────────────────────────────
merged = df_emb[["chunk_size", "overlap_pct", "hit_rate", "mrr"]].merge(
    df_tfidf[["chunk_size", "overlap_pct", "hit_rate", "mrr"]],
    on=["chunk_size", "overlap_pct"], suffixes=("_emb", "_tfidf"),
)
merged["Δhit_rate"] = merged["hit_rate_emb"] - merged["hit_rate_tfidf"]
merged["Δmrr"]      = merged["mrr_emb"]      - merged["mrr_tfidf"]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, delta_col in zip(axes, ["Δhit_rate", "Δmrr"]):
    pivot = merged.pivot_table(values=delta_col, index="chunk_size",
                               columns="overlap_pct")[OVERLAP_ORDER]
    sns.heatmap(pivot, annot=True, fmt="+.3f", cmap="RdYlGn",
                center=0, linewidths=0.5, ax=ax)
    _lbl = "Hit Rate" if "hit" in delta_col else "MRR"
    ax.set_title(f"Δ{_lbl}  (Embeddings − TF-IDF)", fontsize=11, fontweight="bold")
    ax.set_xlabel("Overlap", fontsize=10)
    ax.set_ylabel("Chunk size (tokens)", fontsize=10)

plt.suptitle("Where do semantic embeddings gain / lose vs TF-IDF?",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
_path = FIGURES_DIR / "comparison_delta.png"
fig.savefig(_path, bbox_inches="tight")
plt.show()
print(f"Saved: {_path}")
